# Decision tree regressor tutorial

This notebook walks through **training** and **out-of-sample evaluation** of `mlpackage.supervised_learning.DecisionTreeRegressor`. We use sklearn's Diabetes dataset—numeric features and a continuous progression measure—then hold out test rows and quantify error in multiple ways (`R²`, MAE, RMSE).

## Prerequisites and goals

**You will**
- Load a supervised regression dataset (`X`, numeric `y`).
- Split into **training** vs **held-out test** subsets (no leakage).
- Fit a greedy tree minimizing weighted **variance of targets** across splits (see package README).
- Predict on the unseen test inputs and summarize performance.
- Plot held-out **truth vs predicted** values and compare **`max_depth`** under the same split.

**Dependencies**: `numpy`, `scikit-learn`, `pandas`, `matplotlib` (already in `requirements.txt`).

In [ ]:
from __future__ import annotations

import numpy as np

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from mlpackage.supervised_learning import DecisionTreeRegressor

## Step 1: Load Diabetes regression data

**Diabetes** (sklearn bundles a processed version derived from Bradley et al., 1975) relates tabular patient-derived signals to **disease progression** one year after baseline. Targets are standardized in this particular dataset object; trees still partition on feature thresholds and predict **numeric** responses at leaves.

In [ ]:
diabetes = load_diabetes()
X = np.asarray(diabetes.data)
y = np.asarray(diabetes.target)

feature_names = list(diabetes.feature_names)
target_name = "disease progression (scaled)"

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Target:", target_name)
print("\nFirst rows of X:\n", X[:3])
print("\nFirst y values:\n", y[:10])
print(
    "\ny statistics: mean =",
    round(float(y.mean()), 4),
    "std =",
    round(float(y.std(ddof=0)), 4),
)

## Step 2: Train/test split (out-of-sample holdout)

**Regression**, unlike classification stratification by class frequency, relies on preserving distributional realism with a randomized split. Fixing **`random_state`** keeps notebook runs reproducible.

We never peek at **`X_test` / `y_test`** until after fitting—those rows ground **generalization**, not memorization.

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape[0], "samples")
print("Test:", X_test.shape[0], "samples")

## Step 3: Fit `DecisionTreeRegressor`

Splits maximize **variance drop** weighted by subtree sample sizes—the same heuristic as textbook CART regressors that minimize residual sum of squares heterogeneity locally.

**`max_depth`** controls expressivity: deeper trees memorize training quirks; shallower trees smooth forecasts.

In [ ]:
MAX_DEPTH = 4

reg = DecisionTreeRegressor(max_depth=MAX_DEPTH)
reg.fit(X_train, y_train)

print("Fitted DecisionTreeRegressor with max_depth =", reg.max_depth)

## Step 4: Predict on unseen test inputs

**`predict(X_test)`** returns constant leaf averages learned solely from subsets of **`y_train`**, even though we plot against true held-out \(y_{\mathrm{test}}\) downstairs.

In [ ]:
y_hat = reg.predict(X_test)

print("First 12 predictions:", np.round(y_hat[:12], 4))
print("First 12 true targets:", np.round(y_test[:12], 4))

## Step 5: Metrics on the held-out fold

**`score(X,y)` reports \(R^2\)** (`1 - \mathrm{SSE}/\mathrm{SST}`) identical to sklearn's definition for regressors whenever targets vary.

We additionally print:
- **Mean absolute error (MAE)** – average Absolute residual magnitude.
- **Root mean squared error (RMSE)** – penalizes large mistakes more sharply.

In [ ]:
r2_train = reg.score(X_train, y_train)
r2_test = reg.score(X_test, y_test)

mae_test = np.mean(np.abs(y_test - y_hat))
rmse_test = float(np.sqrt(np.mean((y_test - y_hat) ** 2)))

print(f"Train R^2 : {r2_train:.4f}")
print(f"Test R^2  : {r2_test:.4f}")
print(f"Test MAE  : {mae_test:.4f}")
print(f"Test RMSE : {rmse_test:.4f}")

## Step 6: Inspect individual test residuals

Residual `y_true - y_hat`: negative means **`y_hat` is above** the true target (prediction too high); positive means **`y_hat` is below** the truth (prediction too low).

In [ ]:
n_preview = min(12, X_test.shape[0])
preview_slice = slice(0, n_preview)

df = pd.DataFrame(X_test[preview_slice], columns=feature_names)
df["y_true"] = y_test[preview_slice]
df["y_pred"] = y_hat[preview_slice]
df["residual"] = df["y_true"] - df["y_pred"]
display(df.round(4))

## Step 7: True vs predicted scatter on the test fold

Points hugging the **45° diagonal** indicate agreement; systemic curvature signals bias/variance mismatch worth exploring deeper with residual plots.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_hat, alpha=0.7, edgecolors="black", linewidths=0.3)

lims = float(min(y_test.min(), y_hat.min())), float(max(y_test.max(), y_hat.max()))
margin = (lims[1] - lims[0]) * 0.05 if lims[1] > lims[0] else 1.0
lo, hi = lims[0] - margin, lims[1] + margin
plt.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1, label="y = ŷ")

plt.xlabel("Actual y (held-out)")
plt.ylabel("Predicted ŷ (held-out)")
plt.title("Diabetes: out-of-sample truth vs DecisionTreeRegs")
plt.legend()
plt.axis("square")
plt.tight_layout()

# Also write PNG for non-interactive runs
from pathlib import Path

_nb_here = Path("decision_tree_regressor_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("diabetes_y_vs_yhat_scatter.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/decision_tree_regressor/diabetes_y_vs_yhat_scatter.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
plt.show()

## Step 8: Compare `max_depth` under the **same split**

Watching train \(R^2\) climb alone is misleading—compare it with held-out \(R^2\), MAE, and RMSE whenever capacity increases.

In [ ]:
depth_candidates = [1, 3, 4, None]
rows = []

for md in depth_candidates:
    model = DecisionTreeRegressor(max_depth=md)
    model.fit(X_train, y_train)
    y_te = model.predict(X_test)
    rows.append(
        {
            "max_depth": "None" if md is None else md,
            "train_r2": model.score(X_train, y_train),
            "test_r2": model.score(X_test, y_test),
            "test_mae": float(np.mean(np.abs(y_test - y_te))),
            "test_rmse": float(np.sqrt(np.mean((y_test - y_te) ** 2))),
        }
    )

summary = pd.DataFrame(rows)
display(summary.round(4))